# Team Page — EDA & Data Validation
Validates every data layer the dashboard team page will consume:
- FIFA ranking trajectory
- Form (last 10 results)
- H2H record
- Goals scored / conceded distribution
- WC appearance history

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

PROCESSED = Path('../data/processed')
TEAM = 'England'   # ← change to explore any team

# Load all processed tables
features  = pd.read_parquet(PROCESSED / 'team_features.parquet')
form      = pd.read_parquet(PROCESSED / 'team_form.parquet')
h2h       = pd.read_parquet(PROCESSED / 'team_h2h.parquet')
rankings  = pd.read_parquet(PROCESSED / 'ranking_history.parquet')
goals     = pd.read_parquet(PROCESSED / 'team_goals_dist.parquet')
squads    = pd.read_csv('../data/raw/wc2026_squads.csv')
results   = pd.read_parquet(PROCESSED / 'results_clean.parquet')

print('Loaded all tables.')
print(f'WC 2026 teams in features: {features["is_wc2026_team"].sum()}')

## 1. Team snapshot — top 20 by FIFA rank

In [ ]:
snap = features[features['is_wc2026_team']].sort_values('current_fifa_rank').head(20)
display(snap[[
    'team_name', 'current_fifa_rank', 'current_fifa_points',
    'form_form_win_rate', 'avg_goals_scored_20', 'avg_goals_conceded_20',
    'wc_appearances_approx', 'total_matches_played'
]].rename(columns={
    'form_form_win_rate': 'form_win%',
    'avg_goals_scored_20': 'avg_GF_20',
    'avg_goals_conceded_20': 'avg_GA_20',
}).style.background_gradient(subset=['form_win%','avg_GF_20'], cmap='Greens')
  .background_gradient(subset=['avg_GA_20'], cmap='Reds_r')
  .format(precision=2))

## 2. FIFA ranking trajectory — selected team

In [ ]:
team_rank = rankings[rankings['team_name'] == TEAM].sort_values('rank_date')
print(f"{TEAM}: {len(team_rank)} ranking snapshots, "
      f"{team_rank['rank_date'].min().date()} → {team_rank['rank_date'].max().date()}")

fig = px.line(
    team_rank, x='rank_date', y='rank',
    title=f'{TEAM} — FIFA World Ranking over time',
    labels={'rank_date': 'Date', 'rank': 'FIFA Rank'},
    template='plotly_white'
)
fig.update_yaxes(autorange='reversed', title='FIFA Rank (lower = better)')
fig.update_traces(line_color='#1f77b4', line_width=2)
fig.show()

## 3. Ranking comparison — all WC 2026 teams (latest)

In [ ]:
latest = rankings.sort_values('rank_date').groupby('team_name').last().reset_index()
wc_latest = latest[latest['team_name'].isin(
    features[features['is_wc2026_team']]['team_name']
)].sort_values('rank')

from ingestion.team_name_map import WC2026_TEAMS
wc_latest['confederation'] = wc_latest['team_name'].map(WC2026_TEAMS)

fig = px.bar(
    wc_latest, x='team_name', y='rank',
    color='confederation',
    title='FIFA World Rankings — WC 2026 Qualified Teams',
    labels={'rank': 'FIFA Rank', 'team_name': 'Team'},
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_yaxes(autorange='reversed')
fig.update_xaxes(tickangle=45)
fig.update_layout(height=500)
fig.show()

## 4. Recent form — last 10 results

In [ ]:
team_form = form[form['team_name'] == TEAM].sort_values('date', ascending=False)
print(f'{TEAM} — last {len(team_form)} results:')

def colour_result(val):
    colours = {'W': 'background-color: #c6efce; color: #276221',
               'D': 'background-color: #ffeb9c; color: #9c6500',
               'L': 'background-color: #ffc7ce; color: #9c0006'}
    return colours.get(val, '')

display(
    team_form[['date','opponent','venue','goals_for','goals_against','result','tournament']]
    .style.applymap(colour_result, subset=['result'])
    .format({'date': lambda d: d.strftime('%Y-%m-%d')})
)

## 5. Goals scored / conceded distribution

In [ ]:
team_goals = goals[goals['team_name'] == TEAM].copy()

# Restrict to 2006+ for relevance
team_goals = team_goals[team_goals['date'].dt.year >= 2006]

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Goals Scored per Match', 'Goals Conceded per Match'])

fig.add_trace(go.Histogram(x=team_goals['goals_scored'], nbinsx=10,
                            name='Scored', marker_color='steelblue'), row=1, col=1)
fig.add_trace(go.Histogram(x=team_goals['goals_conceded'], nbinsx=10,
                            name='Conceded', marker_color='salmon'), row=1, col=2)

fig.update_layout(title_text=f'{TEAM} — Goals distribution (2006–present)',
                  template='plotly_white', showlegend=False)
fig.show()

print(f"Mean goals scored:   {team_goals['goals_scored'].mean():.2f}")
print(f"Mean goals conceded: {team_goals['goals_conceded'].mean():.2f}")
print(f"Clean sheets:        {(team_goals['goals_conceded']==0).mean():.1%}")

## 6. H2H record vs WC 2026 opponents

In [ ]:
# Filter h2h pairs involving this team
mask_a = h2h['team_a'] == TEAM
mask_b = h2h['team_b'] == TEAM

h2h_team = pd.concat([
    h2h[mask_a].assign(
        opponent=h2h[mask_a]['team_b'],
        wins=h2h[mask_a]['a_wins'],
        losses=h2h[mask_a]['b_wins'],
        goals_for=h2h[mask_a]['a_goals'],
        goals_against=h2h[mask_a]['b_goals']
    ),
    h2h[mask_b].assign(
        opponent=h2h[mask_b]['team_a'],
        wins=h2h[mask_b]['b_wins'],
        losses=h2h[mask_b]['a_wins'],
        goals_for=h2h[mask_b]['b_goals'],
        goals_against=h2h[mask_b]['a_goals']
    )
], ignore_index=True)

h2h_team = h2h_team[h2h_team['n_played'] > 0][
    ['opponent','n_played','wins','draws','losses','goals_for','goals_against','last_match_date']
].sort_values('n_played', ascending=False)

h2h_team['win_rate'] = (h2h_team['wins'] / h2h_team['n_played']).round(2)
h2h_team['goal_diff'] = h2h_team['goals_for'] - h2h_team['goals_against']

print(f'{TEAM} H2H vs WC 2026 opponents — {len(h2h_team)} matchups:')
display(h2h_team.style.background_gradient(subset=['win_rate'], cmap='RdYlGn')
                      .background_gradient(subset=['goal_diff'], cmap='RdYlGn')
                      .format({'win_rate': '{:.0%}', 'last_match_date': lambda d: str(d)[:10] if pd.notna(d) else '—'}))

## 7. WC appearance history from Fjelstul DB

In [ ]:
fjelstul_matches = pd.read_parquet(PROCESSED / 'fjelstul_matches.parquet')
fjelstul_standings = pd.read_parquet(PROCESSED / 'fjelstul_group_standings.parquet')

# All matches involving this team
team_wc = fjelstul_matches[
    (fjelstul_matches['home_team_name'] == TEAM) |
    (fjelstul_matches['away_team_name'] == TEAM)
].copy()

print(f'{TEAM}: {len(team_wc)} WC matches across {team_wc["tournament_name"].nunique()} tournaments')

# Tournament performances
perf = team_wc.groupby('tournament_name').size().reset_index(name='matches_played')
display(perf)

## 8. Goals scored timeline (rolling 12-month average)

In [ ]:
team_goals_ts = (
    goals[goals['team_name'] == TEAM]
    .set_index('date')
    .sort_index()
    [['goals_scored','goals_conceded']]
)
rolling = team_goals_ts.rolling('365D').mean().reset_index()

fig = px.line(rolling, x='date', y=['goals_scored', 'goals_conceded'],
              title=f'{TEAM} — Rolling 12-month avg goals (scored vs conceded)',
              labels={'value': 'Goals per match', 'date': 'Date'},
              template='plotly_white',
              color_discrete_map={'goals_scored': 'steelblue', 'goals_conceded': 'salmon'})
fig.update_layout(legend_title='')
fig.show()

## 9. Data quality checks

In [ ]:
print('=== COVERAGE CHECK ===')
from ingestion.team_name_map import WC2026_TEAM_NAMES

# Which WC 2026 teams have no ranking history?
ranked_teams = set(rankings['team_name'].unique())
missing_rank = WC2026_TEAM_NAMES - ranked_teams
print(f'WC teams missing FIFA ranking history: {sorted(missing_rank)}')

# Which WC 2026 teams have fewer than 5 form entries?
form_counts = form[form['team_name'].isin(WC2026_TEAM_NAMES)].groupby('team_name').size()
low_form = form_counts[form_counts < 5]
print(f'WC teams with <5 form entries: {low_form.to_dict()}')

# Squad coverage
squads_teams = set(squads['team_name'].unique())
missing_squads = WC2026_TEAM_NAMES - squads_teams
print(f'WC teams missing squad data: {sorted(missing_squads)}')
print(f'Avg squad size: {squads[squads["team_name"].isin(WC2026_TEAM_NAMES)].groupby("team_name").size().mean():.1f}')